# YaadDost: run the Gemma evaluation on Colab
Runs the real card generator against Gemma on Colab's GPU and scores the output. No tunnel or URL needed.

1. **Runtime > Change runtime type > T4 GPU**
2. Run the cells top to bottom. The model download is large and takes a few minutes.
3. The last cell downloads `results.zip`. Bring that file back to the chat.

In [ ]:
# Upload yaaddost-colab.zip when the picker appears
from google.colab import files
import zipfile, os
up = files.upload()
name = next(iter(up))
zipfile.ZipFile(name).extractall("/content/yaaddost")
os.chdir("/content/yaaddost")
print(os.listdir("."))

In [ ]:
%%bash
apt-get install -y -qq zstd > /dev/null
curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1); break
    except Exception:
        time.sleep(1)
print("ollama is up")

In [ ]:
MODEL = "gemma4:e4b"   # for a smaller, faster model use "gemma4:e2b" here AND in the env cell below
!ollama pull {MODEL}

In [ ]:
# Warm up: load the model into GPU memory so the first real request is not slow.
import json, urllib.request
req = urllib.request.Request("http://localhost:11434/api/generate",
    data=json.dumps({"model": MODEL, "prompt": "hi", "stream": False}).encode(),
    headers={"Content-Type": "application/json"})
print(json.load(urllib.request.urlopen(req, timeout=600))["response"][:100])

In [ ]:
import os
os.environ["GEMMA_MODEL"] = MODEL
os.environ["LLM_TIMEOUT"] = "300"
os.makedirs("/content/yaaddost/evals/results", exist_ok=True)
!cd /content/yaaddost && python evals/run_eval.py --language hinglish --count 8 2>&1 | tee evals/results/hinglish.log

In [ ]:
!cd /content/yaaddost && python evals/run_eval.py --language english --count 8 2>&1 | tee evals/results/english.log

In [ ]:
import shutil
shutil.make_archive("/content/results", "zip", "/content/yaaddost/evals/results")
files.download("/content/results.zip")